# 连接到提示中心

我们可以将应用程序连接到 LangSmith 的提示中心，这将允许我们在 LangSmith 内测试和迭代我们的提示词，并将改进直接拉取到我们的应用程序中。

### 设置

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"  # 如果您不设置此项，跟踪将进入默认项目

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

### 从提示中心拉取提示词

通过粘贴 UI 中的代码片段从提示中心拉取提示词。

让我们看看我们拉取了什么 - 注意我们没有获得模型，所以这只是一个 StructuredPrompt 而不是可运行的。

In [ ]:
prompt

很好！现在让我们通过使用输入调用 .invoke() 来构建我们的提示词

In [ ]:
hydrated_prompt = prompt.invoke({"question": "你是船长了吗？", "language": "中文"})
hydrated_prompt

现在让我们将这些消息传递给 OpenAI，看看我们得到什么！

In [ ]:
from openai import OpenAI
from langsmith.client import convert_prompt_to_openai_format

openai_client = OpenAI()

# 注意：我们可以使用 LangSmith 的这个实用程序将我们的构建提示词转换为 openai 格式
converted_messages = convert_prompt_to_openai_format(hydrated_prompt)["messages"]

openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=converted_messages,
    )

##### [额外：仅限 LangChain] 拉取模型配置

当我们使用 `include_model=True` 时，我们也可以将保存的模型配置拉取为 LangChain RunnableBinding。这允许我们直接使用保存的模型配置运行我们的提示模板。

In [ ]:
prompt

测试您的提示词！

In [ ]:
prompt.invoke({"question": "你是船长了吗？", "language": "中文"})

### 拉取特定提交

通过粘贴 UI 中的代码片段从提示中心拉取特定提交。

运行这个提交！

In [ ]:
from openai import OpenAI
from langsmith.client import convert_prompt_to_openai_format

openai_client = OpenAI()

hydrated_prompt = prompt.invoke({"question": "世界是什么样的？", "language": "中文"})
# 注意：我们可以使用 LangSmith 的这个实用程序将我们的构建提示词转换为 openai 格式
converted_messages = convert_prompt_to_openai_format(hydrated_prompt)["messages"]

openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=converted_messages,
    )

### 上传提示词

您也可以轻松地以编程方式更新中心中的提示词。

In [ ]:
from langchain.prompts.chat import ChatPromptTemplate
from langsmith import Client

client=Client()

chinese_prompt = """您是一个问答任务的助手。
使用以下检索到的上下文片段来回答对话中的最新问题。

您的用户只能说中文，请确保您只用中文回答用户。

对话: {conversation}
上下文: {context} 
问题: {question}
答案:"""

chinese_prompt_template = ChatPromptTemplate.from_template(chinese_prompt)
client.push_prompt("chinese-rag-prompt", object=chinese_prompt_template)

您也可以将提示词作为提示词和模型的 RunnableSequence 推送。这对于存储您想要与此提示词一起使用的模型配置很有用。提供商必须受到 LangSmith playground 的支持。

In [ ]:
from langchain.prompts.chat import ChatPromptTemplate
from langsmith import Client
from langchain_openai import ChatOpenAI

client=Client()
model = ChatOpenAI(model="gpt-4o-mini")

chinese_prompt = """您是一个问答任务的助手。
使用以下检索到的上下文片段来回答对话中的最新问题。

您的用户只能说中文，请确保您只用中文回答用户。

对话: {conversation}
上下文: {context} 
问题: {question}
答案:"""
chinese_prompt_template = ChatPromptTemplate.from_template(chinese_prompt)
chain = chinese_prompt_template | model
client.push_prompt("chinese-runnable-sequence", object=chain)